# Analisis Discord GitHub Ecosystem 2026
Notebook disusun mengikuti urutan latihan. **Target modelling: `archived`** (klasifikasi status arsip repository). Simpan CSV di folder yang sama dengan notebook.

## 1. Latihan: Data Loading

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display
from pathlib import Path

# Lokasi CSV relatif ke folder kerja VS Code
DATA_PATH = Path("discord_ecosystem_2026.csv")
if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"File {DATA_PATH} tidak ditemukan. Letakkan CSV di folder kerja VS Code "
        "atau ganti DATA_PATH dengan lokasi lengkap file CSV."
    )
df = pd.read_csv(DATA_PATH)
print("Ukuran dataset:", df.shape)
display(df.head(10))
display(df.tail(5))
print("Kolom:", df.columns.tolist())
df.info()


**Penjelasan baris kode:** `import` memuat pustaka; `DATA_PATH` menentukan nama CSV; `pd.read_csv` membaca CSV; `shape` menghitung baris/kolom; `head`/`tail` menampilkan sampel; `columns` menunjukkan nama fitur; `info` menunjukkan tipe data dan nilai non-null. Simpan CSV di folder kerja VS Code.

## 2. Data Cleaning
Periksa nilai kosong dan duplikat, lalu hilangkan duplikat. Nilai kosong akan ditangani melalui pipeline saat modelling agar tidak terjadi kebocoran data.

In [ ]:
print("Missing values:\n", df.isna().sum().sort_values(ascending=False))
print("Duplikat:", df.duplicated().sum())
df = df.drop_duplicates().copy()
print("Ukuran setelah pembersihan:", df.shape)

**Penjelasan:** `isna().sum()` menghitung nilai hilang per kolom; `duplicated().sum()` menghitung baris duplikat; `drop_duplicates().copy()` menghapus duplikat dan menyalin hasil; `shape` menunjukkan ukuran sesudah pembersihan. Nilai hilang pada fitur akan diisi dalam pipeline setelah splitting untuk menghindari kebocoran data.

## 3. Latihan: Data Cleaning dan Transformation
Konversi tanggal dan buat fitur numerik tambahan tanpa mengubah target.

In [ ]:
for col in ["created_at", "updated_at", "pushed_at"]:
    if col in df.columns:
        df[col] = pd.to_datetime(df[col], errors="coerce", utc=True)
        df[col + "_year"] = df[col].dt.year

for col in ["stars", "forks", "watchers", "open_issues", "stars_per_day", "repo_age_days", "days_since_last_push", "size_kb"]:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

if "stars" in df.columns:
    df["log_stars"] = np.log1p(df["stars"].clip(lower=0))
print(df.dtypes.to_string())
display(df.head())

**Penjelasan:** perulangan `for` memproses kolom hanya jika ada; `to_datetime(..., errors="coerce", utc=True)` mengonversi tanggal dan memberi NaT pada nilai tidak valid; `.dt.year` membuat fitur tahun; `to_numeric` mengonversi angka; `np.log1p` mentransformasi jumlah stars dengan log(1+x) dan `clip` mencegah angka negatif.

## 4. Latihan EDA & ExDA
EDA: ringkasan statistik, distribusi status arsip, dan hubungan antarf fitur. ExDA: pemeriksaan outlier serta perbandingan antarkelompok.

In [ ]:
display(df.describe(include="all").T)
print("Distribusi archived:\n", df["archived"].value_counts(dropna=False))
fig, axes = plt.subplots(1, 2, figsize=(12,4))
sns.countplot(data=df, x="archived", ax=axes[0]); axes[0].set_title("Status arsip")
sns.histplot(data=df, x="stars", bins=35, ax=axes[1]); axes[1].set_title("Distribusi stars")
axes[1].set_yscale("log")
plt.tight_layout(); plt.show()

**Penjelasan:** `describe(include="all")` merangkum statistik; `value_counts` menghitung kelas target; `countplot` menggambar frekuensi status archived; `histplot` memperlihatkan distribusi stars; `set_yscale("log")` membantu membaca rentang frekuensi; `show` menampilkan grafik.

In [ ]:
numeric = df.select_dtypes(include="number")
plt.figure(figsize=(11,7))
sns.heatmap(numeric.corr(), cmap="coolwarm", center=0)
plt.title("Korelasi fitur numerik"); plt.tight_layout(); plt.show()

if "language" in df.columns:
    display(df["language"].fillna("Tidak diketahui").value_counts().head(10).rename("jumlah").to_frame())
if "stars" in df.columns:
    q1, q3 = df["stars"].quantile([.25,.75]); iqr=q3-q1
    print("Outlier stars (aturan IQR):", ((df["stars"] < q1-1.5*iqr) | (df["stars"] > q3+1.5*iqr)).sum())
    sns.boxplot(data=df, x="archived", y="log_stars")
    plt.title("Stars (log) berdasarkan status archived"); plt.show()

**Penjelasan:** `select_dtypes` mengambil fitur numerik; `corr` menghitung korelasi; `heatmap` memvisualisasikannya; `value_counts().head(10)` melihat sepuluh bahasa paling umum; kuartil Q1/Q3 dan IQR digunakan menandai nilai ekstrem; `boxplot` membandingkan distribusi stars setelah transformasi log. Outlier tidak otomatis dihapus.

## 5. Data Processing
Pilih fitur relevan, lalu siapkan preprocessing: imputasi median untuk angka, imputasi modus dan one-hot encoding untuk kategori. Fitting dilakukan **hanya pada data training**.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

target = "archived"
feature_cols = ["stars", "forks", "watchers", "open_issues", "stars_per_day", "repo_age_days", "days_since_last_push", "size_kb", "log_stars", "language", "license", "owner_type", "is_fork", "is_official_discord", "created_at_year"]
feature_cols = [c for c in feature_cols if c in df.columns]
model_df = df.dropna(subset=[target]).copy()
X = model_df[feature_cols].copy()
# Normalisasi target boolean, 0/1, atau teks true/false
y_raw = model_df[target].replace({True: 1, False: 0, "True": 1, "False": 0, "true": 1, "false": 0, "1": 1, "0": 0})
y = pd.to_numeric(y_raw, errors="coerce")
valid = y.isin([0, 1])
X = X.loc[valid].copy()
y = y.loc[valid].astype(int)
if X.empty:
    raise ValueError("Tidak ada label archived valid (0/1 atau True/False).")
num_cols = X.select_dtypes(include="number").columns.tolist()
cat_cols = [c for c in X.columns if c not in num_cols]
X[cat_cols] = X[cat_cols].astype("object").where(X[cat_cols].notna(), np.nan)
preprocessor = ColumnTransformer([
    ("num", Pipeline([("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler())]), num_cols),
    ("cat", Pipeline([("imputer", SimpleImputer(strategy="most_frequent")), ("onehot", OneHotEncoder(handle_unknown="ignore"))]), cat_cols)
])
print("Fitur numerik:", num_cols)
print("Fitur kategori:", cat_cols)
print("Distribusi target:\n", y.value_counts())

**Penjelasan:** `target` adalah label prediksi; `feature_cols` menentukan fitur yang tersedia; `X` adalah fitur dan `y` adalah label; `select_dtypes` memisahkan numerik dan kategorikal; `SimpleImputer` mengisi nilai hilang; `StandardScaler` menstandardisasi numerik; `OneHotEncoder` mengubah kategori menjadi indikator; `ColumnTransformer` menerapkan perlakuan sesuai tipe kolom. Pipeline di-fit pada data training saja.

## 6. Data Splitting
Gunakan 80% training dan 20% testing, dengan stratifikasi bila tiap kelas mempunyai cukup sampel.

In [ ]:
from sklearn.model_selection import train_test_split
if y.nunique() < 2:
    raise ValueError("Target archived hanya memiliki satu kelas. Klasifikasi tidak bisa dilatih.")
stratify = y if y.value_counts().min() >= 2 else None
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=stratify)
print("Training:", X_train.shape, "Testing:", X_test.shape)

**Penjelasan:** `train_test_split` membagi data 80% train dan 20% test; `random_state=42` membuat pembagian konsisten; `stratify=y` mempertahankan proporsi kelas jika memungkinkan; pemeriksaan `nunique` memastikan klasifikasi memiliki dua kelas.

## 7. Latihan: Data Splitting
Verifikasi tidak ada indeks yang tumpang tindih dan lihat proporsi label.

In [ ]:
print("Indeks tumpang tindih:", len(set(X_train.index) & set(X_test.index)))
print("Persentase training:\n", y_train.value_counts(normalize=True).round(3))
print("Persentase testing:\n", y_test.value_counts(normalize=True).round(3))
assert len(X_train) + len(X_test) == len(X)

**Penjelasan:** irisan indeks train/test harus kosong agar tidak ada sampel yang muncul di kedua bagian; `value_counts(normalize=True)` menghitung proporsi label; `assert` memeriksa total sampel tetap sama.

## 8. Pembangunan Model (Modelling)
Random Forest, metrik klasifikasi, confusion matrix, dan perbandingan dengan baseline kelas mayoritas.

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay, balanced_accuracy_score

model = Pipeline([("preprocess", preprocessor), ("classifier", RandomForestClassifier(n_estimators=200, random_state=42, class_weight="balanced"))])
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
print("Akurasi:", round(accuracy_score(y_test, y_pred), 4))
print("Balanced accuracy:", round(balanced_accuracy_score(y_test, y_pred), 4))
print(classification_report(y_test, y_pred, zero_division=0))
ConfusionMatrixDisplay.from_predictions(y_test, y_pred)
plt.title("Confusion Matrix - Random Forest"); plt.show()

baseline = DummyClassifier(strategy="most_frequent").fit(X_train, y_train)
print("Baseline akurasi:", round(accuracy_score(y_test, baseline.predict(X_test)), 4))

**Penjelasan:** `RandomForestClassifier` melatih banyak pohon keputusan; `Pipeline` menyatukan preprocessing dan model; `fit` melatih pada train; `predict` memprediksi test; `accuracy_score` menghitung proporsi prediksi benar; `balanced_accuracy_score` merata-ratakan recall tiap kelas; `classification_report` memberi precision, recall, F1; `ConfusionMatrixDisplay` memvisualisasikan kesalahan; `DummyClassifier` menjadi pembanding kelas mayoritas.

### Kesimpulan
Bandingkan akurasi dan balanced accuracy model dengan baseline. Perhatikan recall dan precision masing-masing kelas, terutama jika data tidak seimbang. Hasil akhir muncul setelah seluruh cell dijalankan.